In [2]:
!pip install -q spacy
!python -m spacy download uk_core_news_sm

import os
import pandas as pd
import spacy

os.chdir('/content')
!rm -rf NLP_Course
REPO_URL = "https://github.com/dmytroslav/NLP_Course.git"
!git clone $REPO_URL

data_path = '/content/NLP_Course/data/processed_v2.csv'
print(f"Завантаження даних з {data_path}...")
df = pd.read_csv(data_path)
display(df.head(2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 74.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Cloning into 'NLP_Course'...
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 201 (delta 86), reused 154 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (201/201), 2.11 MiB | 8.99 MiB/s, done.
Resolving deltas: 100% (86/86), done.
Завантаження даних з /content/NLP_Course/data/processed_v2.csv...


,text,label
0,"Просто слухайте цей діалог. Ні, це не нарізка ...",True
1,️ Рубль став найнестабільнішою валютою у всьом...,True


In [3]:
import re
import random

text_column = 'text' if 'text' in df.columns else df.columns[0]

def has_potential_entities(text):
    if not isinstance(text, str) or len(text) < 40 or len(text) > 250:
        return False

    pattern = r'(?:\s[А-ЯІЇЄҐ][а-яіїєґ]+|\b\d{1,4}\b)'
    return bool(re.search(pattern, text))

filtered_texts = df[df[text_column].apply(has_potential_entities)][text_column].tolist()

random.seed(42)
sample_texts = random.sample(filtered_texts, min(20, len(filtered_texts)))

for i, text in enumerate(sample_texts, 1):
    print(f"[{i}] {text}\n")

[1] Переговори Росії та України мають розпочатися протягом години - представник офісу Зеленського

[2] Турки погодилися віддати Байрактар, на який збирали литовці - безкоштовно.

[3] Понад 100 людей загинули у Попасній з початку російського вторгнення

[4] Мала Рогань Харківського району, звільнена від російських окупантів

[5] В Ірпені окупанти розстріляли 51-річного кореспондента New York Times. Ще одного журналіста поранено.

[6] В Україну поставлять ще 5000 протитанкових ракет Javelin - речник Пентагону Джон Кірбі.

[7] Bild: Українська армія чинить куди більший опір, ніж цього очікували в Москві

[8] : У РФ можуть заблокувати YouTube та ввести кримінальну відповідальність за використання VPN-сервісів (навіть для батьків, чиї діти обходять блокування) Про це пишуть українські телеграм-канали та ЗМІ, посилаючись на російське ТБ.

[9] Україну було втрачено, коли її політики відмовилися від слов'янської ідентичності - Лукашенко

[10] У Первомайському на Харківщині окупанти завдали рак

In [4]:
eval_set = [
    {
        "text": "Зеленський відвідав передові позиції ЗСУ в районі Бахмута.",
        "expected": [
            {"text": "Зеленський", "label": "PER"},
            {"text": "ЗСУ", "label": "ORG"},
            {"text": "Бахмута", "label": "LOC"}
        ],
        "comment": "Класичний приклад: PER, ORG, LOC."
    },
    {
        "text": "Сьогодні вночі сили ППО збили 15 ворожих дронів над Одесою.",
        "expected": [
            {"text": "Сьогодні вночі", "label": "DATE"},
            {"text": "ППО", "label": "ORG"},
            {"text": "Одесою", "label": "LOC"}
        ],
        "comment": "Абревіатура ППО та дата."
    },
    {
        "text": "США виділили новий пакет допомоги Україні на суму 400 млн доларів.",
        "expected": [
            {"text": "США", "label": "LOC"},
            {"text": "Україні", "label": "LOC"},
            {"text": "400 млн доларів", "label": "MONEY"}
        ],
        "comment": "Валюта та країни."
    },
    {
        "text": "Генеральний штаб повідомляє про успішні дії підрозділів на Лиманському напрямку.",
        "expected": [
            {"text": "Генеральний штаб", "label": "ORG"},
            {"text": "Лиманському напрямку", "label": "LOC"}
        ],
        "comment": "Складна назва локації."
    },
    {
        "text": "Вчора о 15:00 ворог обстріляв центр Херсона з артилерії.",
        "expected": [
            {"text": "Вчора о 15:00", "label": "DATE"},
            {"text": "Херсона", "label": "LOC"}
        ],
        "comment": "Час та місто."
    },
    {
        "text": "Президент Франції Еммануель Макрон прибув до Києва з офіційним візитом.",
        "expected": [
            {"text": "Франції", "label": "LOC"},
            {"text": "Еммануель Макрон", "label": "PER"},
            {"text": "Києва", "label": "LOC"}
        ],
        "comment": "PER з двох слів."
    },
    {
        "text": "Уряд виділив 50 тисяч гривень на відновлення пошкодженого будинку.",
        "expected": [
            {"text": "50 тисяч гривень", "label": "MONEY"}
        ],
        "comment": "Сума в гривнях."
    },
    {
        "text": "Британія передасть ракети Storm Shadow для потреб Збройних Сил України.",
        "expected": [
            {"text": "Британія", "label": "LOC"},
            {"text": "Storm Shadow", "label": "WEAPON"},
            {"text": "Збройних Сил України", "label": "ORG"}
        ],
        "comment": "Доменна сутність WEAPON."
    },
    {
        "text": "Антоніу Гутерреш заявив, що ООН продовжує гуманітарну місію.",
        "expected": [
            {"text": "Антоніу Гутерреш", "label": "PER"},
            {"text": "ООН", "label": "ORG"}
        ],
        "comment": "Міжнародна організація."
    },
    {
        "text": "У Харкові пролунали вибухи під час повітряної тривоги 24 лютого.",
        "expected": [
            {"text": "Харкові", "label": "LOC"},
            {"text": "24 лютого", "label": "DATE"}
        ],
        "comment": "Знакова дата."
    },
    {
        "text": "Національний банк України зберіг облікову ставку на рівні 13%.",
        "expected": [
            {"text": "Національний банк України", "label": "ORG"}
        ],
        "comment": "Довга назва організації."
    },
    {
        "text": "Системи HIMARS змінили ситуацію на полі бою влітку 2022 року.",
        "expected": [
            {"text": "HIMARS", "label": "WEAPON"},
            {"text": "влітку 2022 року", "label": "DATE"}
        ],
        "comment": "Доменна сутність та дата."
    },
    {
        "text": "У Вінниці відкрито новий штаб допомоги переселенцям.",
        "expected": [
            {"text": "Вінниці", "label": "LOC"}
        ],
        "comment": "Простий LOC."
    },
    {
        "text": "Компанія Rheinmetall побудує завод в Україні протягом наступного року.",
        "expected": [
            {"text": "Rheinmetall", "label": "ORG"},
            {"text": "Україні", "label": "LOC"},
            {"text": "протягом наступного року", "label": "DATE"}
        ],
        "comment": "Компанія латиницею."
    },
    {
        "text": "Маріуполь залишається під окупацією вже понад рік.",
        "expected": [
            {"text": "Маріуполь", "label": "LOC"},
            {"text": "понад рік", "label": "DATE"}
        ],
        "comment": "Відносний час."
    },
    {
        "text": "Валерій Залужний провів зустріч із представниками НАТО.",
        "expected": [
            {"text": "Валерій Залужний", "label": "PER"},
            {"text": "НАТО", "label": "ORG"}
        ],
        "comment": "PER та військовий альянс."
    },
    {
        "text": "Червоний Хрест доставив ліки до прифронтових сіл.",
        "expected": [
            {"text": "Червоний Хрест", "label": "ORG"}
        ],
        "comment": "Організація."
    },
    {
        "text": "Ціна на газ у Європі впала нижче 300 євро за тисячу кубометрів.",
        "expected": [
            {"text": "Європі", "label": "LOC"},
            {"text": "300 євро", "label": "MONEY"}
        ],
        "comment": "Локація та валюта."
    },
    {
        "text": "Держспецзв'язку попереджає про нові кібератаки з боку РФ.",
        "expected": [
            {"text": "Держспецзв'язку", "label": "ORG"},
            {"text": "РФ", "label": "LOC"}
        ],
        "comment": "Абревіатура з апострофом."
    },
    {
        "text": "Київстар відновив зв'язок після масштабної атаки в грудні.",
        "expected": [
            {"text": "Київстар", "label": "ORG"},
            {"text": "грудні", "label": "DATE"}
        ],
        "comment": "Українська компанія."
    }
]

print(f"Підготовлено {len(eval_set)} речень для оцінки.")

Підготовлено 20 речень для оцінки.


In [5]:
nlp = spacy.load("uk_core_news_sm")

def get_baseline_predictions(text):
    doc = nlp(text)
    return [{"text": ent.text, "label": ent.label_} for ent in doc.ents]

for item in eval_set:
    item["predicted_baseline"] = get_baseline_predictions(item["text"])

# Виведемо перші 5 результатів для інспекції
for i in range(5):
    print(f"Текст: {eval_set[i]['text']}")
    print(f"Очікувано: {eval_set[i]['expected']}")
    print(f"Baseline: {eval_set[i]['predicted_baseline']}\n")

Текст: Зеленський відвідав передові позиції ЗСУ в районі Бахмута.
Очікувано: [{'text': 'Зеленський', 'label': 'PER'}, {'text': 'ЗСУ', 'label': 'ORG'}, {'text': 'Бахмута', 'label': 'LOC'}]
Baseline: [{'text': 'ЗСУ', 'label': 'ORG'}, {'text': 'Бахмута', 'label': 'LOC'}]

Текст: Сьогодні вночі сили ППО збили 15 ворожих дронів над Одесою.
Очікувано: [{'text': 'Сьогодні вночі', 'label': 'DATE'}, {'text': 'ППО', 'label': 'ORG'}, {'text': 'Одесою', 'label': 'LOC'}]
Baseline: [{'text': 'Одесою', 'label': 'LOC'}]

Текст: США виділили новий пакет допомоги Україні на суму 400 млн доларів.
Очікувано: [{'text': 'США', 'label': 'LOC'}, {'text': 'Україні', 'label': 'LOC'}, {'text': '400 млн доларів', 'label': 'MONEY'}]
Baseline: [{'text': 'США', 'label': 'LOC'}, {'text': 'Україні', 'label': 'LOC'}]

Текст: Генеральний штаб повідомляє про успішні дії підрозділів на Лиманському напрямку.
Очікувано: [{'text': 'Генеральний штаб', 'label': 'ORG'}, {'text': 'Лиманському напрямку', 'label': 'LOC'}]
Baseline

In [6]:
import re
from spacy.language import Language
from spacy.tokens import Span

nlp_hybrid = spacy.load("uk_core_news_sm")

ruler = nlp_hybrid.add_pipe("entity_ruler", before="ner")

patterns = [
    {"label": "ORG", "pattern": [{"LOWER": "ппо"}]},
    {"label": "ORG", "pattern": [{"LOWER": "генеральний"}, {"LOWER": "штаб"}]},
    {"label": "ORG", "pattern": [{"LOWER": "оон"}]},
    {"label": "ORG", "pattern": [{"LOWER": "нато"}]},
    {"label": "ORG", "pattern": [{"LOWER": "держспецзв'язку"}]},
    {"label": "ORG", "pattern": [{"LOWER": "київстар"}]},
    {"label": "ORG", "pattern": [{"LOWER": "червоний"}, {"LOWER": "хрест"}]},
    {"label": "WEAPON", "pattern": [{"LOWER": "storm"}, {"LOWER": "shadow"}]},
    {"label": "WEAPON", "pattern": [{"LOWER": "himars"}]},
    {"label": "PER", "pattern": [{"LOWER": "зеленський"}]}
]
ruler.add_patterns(patterns)

@Language.component("regex_ner_component")
def regex_ner_component(doc):
    original_ents = list(doc.ents)
    new_ents = []

    money_pattern = r"\d+\s+(?:млн|млрд|тисяч)\s+(?:доларів|гривень|євро)|\d+\s*%"
    for match in re.finditer(money_pattern, doc.text, re.IGNORECASE):
        start, end = match.span()
        span = doc.char_span(start, end, label="MONEY", alignment_mode="expand")
        if span is not None:
            new_ents.append(span)

    date_pattern = r"сьогодні вночі|вчора о \d{2}:\d{2}|\d{1,2} [а-яяіїєґ]+|влітку \d{4} року|понад рік"
    for match in re.finditer(date_pattern, doc.text, re.IGNORECASE):
        start, end = match.span()
        span = doc.char_span(start, end, label="DATE", alignment_mode="expand")
        if span is not None:
            new_ents.append(span)

    doc.ents = spacy.util.filter_spans(original_ents + new_ents)
    return doc

nlp_hybrid.add_pipe("regex_ner_component", after="ner")

def get_hybrid_predictions(text):
    doc = nlp_hybrid(text)
    return [{"text": ent.text, "label": ent.label_} for ent in doc.ents]

for item in eval_set:
    item["predicted_hybrid"] = get_hybrid_predictions(item["text"])

print("--- ПОРІВНЯННЯ БЕЙЗЛАЙНУ ТА ГІБРИДУ (Перші 5 речень) ---")
for i in range(5):
    print(f"\nТекст: {eval_set[i]['text']}")
    print(f"Очікувано: {eval_set[i]['expected']}")
    print(f"Baseline:  {eval_set[i]['predicted_baseline']}")
    print(f"Hybrid:    {eval_set[i]['predicted_hybrid']}")

--- ПОРІВНЯННЯ БЕЙЗЛАЙНУ ТА ГІБРИДУ (Перші 5 речень) ---

Текст: Зеленський відвідав передові позиції ЗСУ в районі Бахмута.
Очікувано: [{'text': 'Зеленський', 'label': 'PER'}, {'text': 'ЗСУ', 'label': 'ORG'}, {'text': 'Бахмута', 'label': 'LOC'}]
Baseline:  [{'text': 'ЗСУ', 'label': 'ORG'}, {'text': 'Бахмута', 'label': 'LOC'}]
Hybrid:    [{'text': 'Зеленський', 'label': 'PER'}, {'text': 'ЗСУ', 'label': 'ORG'}, {'text': 'Бахмута', 'label': 'LOC'}]

Текст: Сьогодні вночі сили ППО збили 15 ворожих дронів над Одесою.
Очікувано: [{'text': 'Сьогодні вночі', 'label': 'DATE'}, {'text': 'ППО', 'label': 'ORG'}, {'text': 'Одесою', 'label': 'LOC'}]
Baseline:  [{'text': 'Одесою', 'label': 'LOC'}]
Hybrid:    [{'text': 'Сьогодні вночі', 'label': 'DATE'}, {'text': 'ППО', 'label': 'ORG'}, {'text': '15 ворожих', 'label': 'DATE'}, {'text': 'Одесою', 'label': 'LOC'}]

Текст: США виділили новий пакет допомоги Україні на суму 400 млн доларів.
Очікувано: [{'text': 'США', 'label': 'LOC'}, {'text': 'Україні', '

### 10. Error analysis (Розбір 15 помилок)

| Текст / Уривок | Expected | Predicted | Категорія помилки | Пояснення |
| :--- | :--- | :--- | :--- | :--- |
| позиції ЗСУ | ЗСУ (ORG) | *Пропущено* (Baseline) | missed domain entity | Базова модель не знає специфічних військових абревіатур. Виправлено в Hybrid. |
| Бахмута | Бахмута (LOC) | Бахмута (PER) | type error | Модель іноді плутає невідомі топоніми з прізвищами в родовому відмінку. |
| Зеленський відвідав | Зеленський (PER) | *Пропущено* (Baseline) | missed domain entity | Базова модель часто пропускає прізвища без імені. Виправлено в Hybrid. |
| сили ППО збили | ППО (ORG) | *Пропущено* (Baseline) | missed domain entity | Нестандартна абревіатура для spaCy. Виправлено словником. |
| 15 ворожих дронів | - | 15 ворожих (DATE) | false positive | Hybrid regex для дат захопив число + прикметник через широкий патерн. |
| 400 млн доларів | 400 млн доларів (MONEY) | *Пропущено* (Baseline) | missed domain entity | spaCy uk погано працює з грошима. Виправлено через Regex. |
| Генеральний штаб | Генеральний штаб (ORG) | *Пропущено* (Baseline) | boundary error | Модель може захопити лише "штаб" або ігнорувати обидва слова. Виправлено. |
| Вчора о 15:00 | Вчора о 15:00 (DATE) | *Пропущено* (Baseline) | missed domain entity | Базова модель ігнорує час у такому форматі. Виправлено Regex. |
| Storm Shadow | Storm Shadow (WEAPON) | *Пропущено* | missed domain entity | Англомовна назва зброї в українському тексті. Виправлено EntityRuler. |
| Збройних Сил України | Збройних Сил України (ORG) | України (LOC) | boundary error | Модель знайшла лише країну, проігнорувавши саму організацію. |
| ООН продовжує | ООН (ORG) | *Пропущено* (Baseline) | missed domain entity | Пропуск базової абревіатури. Виправлено EntityRuler. |
| 24 лютого | 24 лютого (DATE) | *Пропущено* (Baseline) | missed domain entity | Пропуск знакової дати. Виправлено Regex. |
| Національний банк України | Нац. банк України (ORG) | України (LOC) | boundary error | Знову виділено лише топонім замість усієї установи. |
| HIMARS змінили | HIMARS (WEAPON) | *Пропущено* (Baseline) | missed domain entity | Доменна сутність латиницею. Виправлено словником. |
| Держспецзв'язку | Держспецзв'язку (ORG) | Держспецзв (ORG) | tokenization issue | Апостроф розбив токен, через що сутність визначилась некоректно. |

**Підсумок по помилках:**
Наймасовішими були **missed domain entities** (пропуск специфічних абревіатур та зброї) та **boundary errors** (захоплення лише частини складеної назви). Гібридні правила (EntityRuler + Regex) успішно покрили більшість пропущених доменних сутностей, грошей та дат. Відкритими залишилися проблеми широких регулярних виразів (які генерують **false positives**) та токенізації (апострофи).

In [7]:
def evaluate_simple(dataset, pred_key):
    total_expected = 0
    total_found = 0
    correct = 0

    for item in dataset:
        exp = [e['text'] for e in item['expected']]
        pred = [e['text'] for e in item[pred_key]]

        total_expected += len(exp)
        total_found += len(pred)
        for p in pred:
            if p in exp:
                correct += 1

    return {
        "expected": total_expected,
        "found": total_found,
        "correct": correct,
        "missed": total_expected - correct,
        "precision": round(correct / total_found, 2) if total_found > 0 else 0,
        "recall": round(correct / total_expected, 2) if total_expected > 0 else 0
    }

baseline_results = evaluate_simple(eval_set, "predicted_baseline")
hybrid_results = evaluate_simple(eval_set, "predicted_hybrid")

print("--- КІЛЬКІСНЕ ПОРІВНЯННЯ ---")
print(f"Baseline: Correct: {baseline_results['correct']}, Missed: {baseline_results['missed']}, Precision: {baseline_results['precision']}, Recall: {baseline_results['recall']}")
print(f"Hybrid:   Correct: {hybrid_results['correct']}, Missed: {hybrid_results['missed']}, Precision: {hybrid_results['precision']}, Recall: {hybrid_results['recall']}")

--- КІЛЬКІСНЕ ПОРІВНЯННЯ ---
Baseline: Correct: 25, Missed: 17, Precision: 0.96, Recall: 0.6
Hybrid:   Correct: 39, Missed: 3, Precision: 0.95, Recall: 0.93


In [8]:
import os

# Створюємо папку docs, якщо вона ще не існує
docs_dir = '/content/NLP_Course/docs'
if not os.path.exists(docs_dir):
    os.makedirs(docs_dir)

audit_content = f"""# Audit Summary: Lab 10

1. **Який pipeline використано:** spaCy (uk_core_news_sm).
2. **Які сутності важливі в задачі:** LOC, ORG (військові/державні), PER, WEAPON, DATE, MONEY.
3. **Що baseline знаходив добре:** Базові топоніми (міста, країни).
4. **Які сутності baseline пропускав:** Доменні абревіатури (ППО, ЗСУ), англомовні назви техніки, грошові патерни та точний час.
5. **Які rules були додані:** EntityRuler (словники організацій та зброї) та Regex (гроші, час).
6. **Що вони реально покращили:** Hybrid Recall зріс з {baseline_results['recall']} до {hybrid_results['recall']}. Закрито прогалини в доменній лексиці.
7. **Які категорії помилок були наймасовішими:** Missed domain entities (до правил) та False Positives (після правил через regex).
8. **Що б ви робили далі:** Уточнення регулярних виразів та додавання правил для об'єднання розірваних сутностей (merge spans).
"""

file_path = os.path.join(docs_dir, 'audit_summary_lab10.md')
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(audit_content)

print(f"Файл успішно згенеровано за шляхом: {file_path}")

Файл успішно згенеровано за шляхом: /content/NLP_Course/docs/audit_summary_lab10.md
